# Swipe Atlas Final Workflow

Main project story: predict dating-app engagement (`mutual_matches`) and segment users by behavior. `match_outcome` is retained as a no-signal case study because the synthetic labels are balanced and not practically predictable from the available features.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
print(ROOT)

## 1. EDA and Signal Check

In [ ]:
subprocess.run([sys.executable, "scripts/01_eda.py"], check=True)
subprocess.run([sys.executable, "scripts/02_signal_test.py"], check=True)

## 2. Engagement Modeling

The official model predicts `mutual_matches` without using `likes_received` or `match_outcome`. A paired-feature comparison is generated separately to show leakage inflation.

In [ ]:
subprocess.run([sys.executable, "scripts/04_train_engagement_models.py"], check=True)

## 3. User Segmentation

In [ ]:
subprocess.run([sys.executable, "scripts/05_segmentation.py"], check=True)

## 4. AutoML Comparison (Rubric Step 7)

> ⚠️ **Run this section in Google Colab (Linux) only.** auto-sklearn and AutoGluon require Linux.  
> On Windows/Python 3.12 skip this cell and paste the Colab output into the report.
>
> **Why it matters:** the rubric (Step 7) explicitly asks "How does your model compare to auto-sklearn?"  
> An AutoML system that searches thousands of model–hyperparameter combinations and still lands at R²≈0  
> is the most rhetorically powerful evidence that the low signal is a data property, not a modeling failure.

### Option A — auto-sklearn 2.0 (rubric-compliant citation)
Reference: Feurer, Eggensperger, Falkner, Lindauer & Hutter, JMLR 23(261):1–61, 2022

### Option B — AutoGluon 1.x (2026 SOTA, best fallback if auto-sklearn install fails)
Reference: Gijsbers et al., JMLR 25(101):1–65, 2024 (OpenML AutoML Benchmark)

In [ ]:
# ============================================================
# OPTION A: auto-sklearn 2.0  (rubric Step 7 compliance)
# Run in Google Colab (Linux). Skip on Windows.
# ============================================================

# Step 1: Install
# !pip install auto-sklearn

# Step 2: Run
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("dating_app_behavior_dataset_extended1.csv")

# Same safe feature set as official model (excludes likes_received, match_outcome)
drop_cols = ["match_outcome", "mutual_matches", "likes_received", "interest_tags"]
X = pd.get_dummies(df.drop(columns=drop_cols), drop_first=True)
y = df["mutual_matches"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from autosklearn.regression import AutoSklearnRegressor
automl_a = AutoSklearnRegressor(time_left_for_this_task=300, per_run_time_limit=60, seed=42)
automl_a.fit(X_train, y_train)

preds_a = automl_a.predict(X_test)
r2_a  = r2_score(y_test, preds_a)
mae_a = mean_absolute_error(y_test, preds_a)
print(f"auto-sklearn  R²={r2_a:.3f}  MAE={mae_a:.3f}")
print(automl_a.leaderboard())

# ============================================================
# OPTION B: AutoGluon 1.x  (2026 SOTA, use if auto-sklearn fails)
# ============================================================

# Step 1: Install
# !pip install autogluon.tabular

# Step 2: Run
from autogluon.tabular import TabularDataset, TabularPredictor

train_data = TabularDataset(pd.concat([X_train, y_train], axis=1))
test_data  = TabularDataset(pd.concat([X_test,  y_test],  axis=1))

predictor = TabularPredictor(label="mutual_matches", eval_metric="r2").fit(
    train_data, presets="best_quality", time_limit=3600
)

leaderboard = predictor.leaderboard(test_data, silent=True)
print(leaderboard[["model", "score_test", "score_val"]].to_string())

# ============================================================
# Comparison table (fill from output, paste into report)
# ============================================================
print("\n--- COMPARISON TABLE ---")
print(f"{'Model':<30} {'R²':>8} {'Notes'}")
print(f"{'Dummy mean (baseline)':<30} {'≈0.000':>8}  safe feature set")
print(f"{'Best manual (Hist GB tuned)':<30} {'≈-0.003':>8}  safe feature set")
print(f"{'auto-sklearn 2.0':<30} {r2_a:>8.3f}  300s budget, safe feature set")
print("AutoGluon 1.x: see leaderboard above")
print("\nExpected: all land near R²=0 — confirming low signal is a data property, not a modeling failure.")

## 5. Report Tables

In [ ]:
import pandas as pd
display(pd.read_csv("reports/engagement_model_results.csv"))
display(pd.read_csv("reports/segmentation_summary.csv"))